# Small Reasoning LLM Lab — Evaluation

**How to use:**
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all`
3. When prompted, allow Google Drive access
4. All results are printed automatically

The notebook finds `best.pt` automatically — first in Google Drive (where
the training notebook saves it), then anywhere else in the session.
If nothing is found, it offers to train the model.

**You do not need to type any file paths.**

## Step 1 — Setup: Drive, repo, dependencies

In [ ]:
import os, sys, subprocess, glob

# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/small-llm-lab'
print(f'Drive root: {DRIVE_ROOT}')

# ── Clone / update repo (code only — no checkpoints in git) ──────────────────
REPO_URL = 'https://github.com/sinor77/small-llm-lab.git'
CODE_DIR = '/content/small-llm-lab'

def _git(*args, cwd=None):
    r = subprocess.run(['git'] + list(args), cwd=cwd or os.getcwd(),
                       capture_output=True, text=True)
    for line in (r.stdout + r.stderr).strip().splitlines():
        print(f'  git: {line}')

if not os.path.exists(CODE_DIR):
    print('Cloning repository...')
    _git('clone', REPO_URL, cwd='/content')
else:
    print('Pulling latest code...')
    _git('pull', 'origin', 'main', cwd=CODE_DIR)

os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
print(f'Code directory: {os.getcwd()}')
_git('log', '--oneline', '-3', cwd=CODE_DIR)

# ── Install tokenizers if needed ──────────────────────────────────────────────
try:
    import tokenizers
    print(f'tokenizers {tokenizers.__version__} already installed.')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    'tokenizers>=0.15.0', '-q'], check=True)
    print('tokenizers installed.')

## Step 2 — Detect GPU

In [ ]:
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    print(f'GPU:    {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('No GPU — evaluation will be slow. Consider Runtime -> T4 GPU.')
print(f'Device: {DEVICE}')

## Step 3 — Find best.pt (searches Drive, then local, then offers to train)

In [ ]:
import json, time

BEST_CK = None
EXP_DIR = None
CK_DIR  = None

def _newest(pattern):
    hits = glob.glob(pattern, recursive=True)
    hits.sort(key=os.path.getmtime, reverse=True)
    return hits

def _set_paths(p):
    global BEST_CK, CK_DIR, EXP_DIR
    BEST_CK = os.path.abspath(p)
    CK_DIR  = os.path.dirname(BEST_CK)
    # Walk up to find tokenizer/ sibling = experiment root
    candidate = CK_DIR
    for _ in range(5):
        if os.path.isdir(os.path.join(candidate, 'tokenizer')):
            EXP_DIR = candidate
            return
        candidate = os.path.dirname(candidate)
    EXP_DIR = os.path.dirname(CK_DIR)

# 1. Google Drive (primary — training notebook saves here)
print('Searching Google Drive...')
hits = _newest(os.path.join(DRIVE_ROOT, '**/best.pt'))
if hits:
    _set_paths(hits[0])
    print(f'  Found on Drive: {BEST_CK}')
    if len(hits) > 1:
        print(f'  ({len(hits)} experiments on Drive — using newest)')
        for h in hits: print(f'    {h}')

# 2. experiments/results/ inside the repo (local, this session)
if not BEST_CK:
    print('Not found on Drive. Searching local experiments/results/ ...')
    hits = _newest(os.path.join(CODE_DIR, 'experiments', 'results',
                                '*', 'checkpoints', 'best.pt'))
    if hits:
        _set_paths(hits[0])
        print(f'  Found locally: {BEST_CK}')

# 3. Anywhere in /content
if not BEST_CK:
    print('Searching all of /content ...')
    hits = _newest('/content/**/best.pt')
    if hits:
        _set_paths(hits[0])
        print(f'  Found: {BEST_CK}')

if BEST_CK:
    print()
    print('=' * 55)
    print(f'  Checkpoint: {BEST_CK}')
    print(f'  Exp dir:    {EXP_DIR}')
    print('=' * 55)
else:
    print()
    print('=' * 55)
    print('  NO CHECKPOINT FOUND ANYWHERE')
    print('  best.pt is not on Drive or in this session.')
    print('  The next cell will train the model (~10-30 min).')
    print('  After training, checkpoints are saved to Drive.')
    print('=' * 55)

In [ ]:
# ── Train if no checkpoint found ──────────────────────────────────────────────
# If best.pt was found above this cell does nothing and completes instantly.
# If not found, it runs training and saves checkpoints to Google Drive.

if BEST_CK is None:
    print('Training the model now. Checkpoints will be saved to Google Drive.')
    print('This takes approximately 10-30 minutes on a T4 GPU.')
    print()

    from training.train import train
    from training.config import get_train_config_by_name

    tc = get_train_config_by_name('colab_small')

    # Redirect output to Drive
    DRIVE_EXP = os.path.join(DRIVE_ROOT, tc.output_dir)
    os.makedirs(DRIVE_EXP, exist_ok=True)
    tc.output_dir        = DRIVE_EXP
    tc.checkpoint_dir    = os.path.join(DRIVE_EXP, 'checkpoints')
    tc.tokenizer_dir     = os.path.join(DRIVE_EXP, 'tokenizer')
    tc.metrics_file      = os.path.join(DRIVE_EXP, 'metrics.jsonl')
    tc.model_config_file = os.path.join(DRIVE_EXP, 'model_config.json')
    tc.train_config_file = os.path.join(DRIVE_EXP, 'train_config.json')
    tc.eval_file         = os.path.join(DRIVE_EXP, 'evaluation.json')
    tc.samples_file      = os.path.join(DRIVE_EXP, 'samples.json')

    train(tc)

    new_best = os.path.join(tc.checkpoint_dir, 'best.pt')
    if os.path.exists(new_best):
        _set_paths(new_best)
        print(f'Training complete. Checkpoint saved to: {BEST_CK}')
    else:
        raise RuntimeError('Training finished but best.pt was not created. '
                           'Check the training output above for errors.')
else:
    print(f'Checkpoint already found — skipping training.\n{BEST_CK}')

## Step 4 — Load model

In [ ]:
from inference.generate import load_model_and_tokenizer, generate_answer, solve_problem
from data.generators.arithmetic import ArithmeticGenerator
from evaluation.benchmark import run_benchmark, run_generalization_benchmark
from evaluation.arithmetic import verify

model, tokenizer, ck = load_model_and_tokenizer(BEST_CK, device=DEVICE)
n_params = model.count_parameters()
step     = ck.get('global_step', '?')
cfg      = model.config

print('=== Model loaded ===')
print(f'  Checkpoint:    {BEST_CK}')
print(f'  Parameters:    {n_params:,}')
print(f'  Training step: {step}')
print(f'  Architecture:  d{cfg.d_model}/L{cfg.n_layers}/H{cfg.n_heads}/ff{cfg.d_ff}')
print(f'  Tokenizer:     {tokenizer.vocab_size} tokens')
print(f'  Device:        {DEVICE}')

generator = ArithmeticGenerator(seed=42, difficulty='medium')

## Step 5 — Test benchmark (500 unseen problems)

In [ ]:
print('Running test benchmark (~2-3 min on T4)...')
t0 = time.time()
bench = run_benchmark(
    model=model, tokenizer=tokenizer, generator=generator,
    n_problems=500, split='test', max_new_tokens=64, temperature=0.0,
    use_reasoning=True, max_samples_to_save=50, device=DEVICE,
)
print(f'Done in {time.time()-t0:.0f}s\n')
print(bench.summary_str())

## Step 6 — Generalization benchmark (5 difficulty levels)

In [ ]:
print('Building exclusion set from training data...')
train_ex, val_ex, test_ex = generator.generate_all_splits(
    n_train=20000, n_val=2000, n_test=1000)
all_known = set(e.problem for e in train_ex + val_ex + test_ex)
print(f'Exclusion set: {len(all_known):,} problems')
print()
print('Running generalization benchmark (~5-8 min on T4)...')
t0 = time.time()
gen_results = run_generalization_benchmark(
    model=model, tokenizer=tokenizer, generator=generator,
    n_per_level=200, max_new_tokens=64, temperature=0.0,
    device=DEVICE, use_reasoning=True, exclude_problems=all_known,
)
print(f'Done in {time.time()-t0:.0f}s\n')

LEVEL_DESC = {
    1: 'New values, same templates (medium)',
    2: 'Larger numbers (hard difficulty)',
    3: 'Mixed families (medium)',
    4: 'Long chains — multi_step + algebra (hard)',
    5: 'All families, hard difficulty',
}
print(f'{"Lvl":<5}{"Correct":>8}{"Total":>7}{"Acc":>8}  Description')
print('-' * 65)
tc, tn = 0, 0
for lv in sorted(gen_results):
    r = gen_results[lv]
    tc += r['correct']; tn += r['total']
    print(f"{lv:<5}{r['correct']:>8}{r['total']:>7}{r['accuracy']:>8.1%}  {LEVEL_DESC[lv]}")
print('-' * 65)
print(f"{'ALL':<5}{tc:>8}{tn:>7}{tc/tn:>8.1%}  Overall")

# Save to Drive
gen_out = os.path.join(EXP_DIR, 'generalization_results.json')
os.makedirs(EXP_DIR, exist_ok=True)
with open(gen_out, 'w') as f: json.dump(gen_results, f, indent=2)
print(f'\nSaved to: {gen_out}')

## Step 7 — Checkpoint progression

In [ ]:
PROBE = [
    ('single_op',    'What is 37 + 58?',                                                            '95'),
    ('comparison',   'Which is greater: 44 or 51?',                                                 '51'),
    ('number_seq',   'What is the next number in the sequence: 2, 4, 8, 16, 32, ...?',              '64'),
    ('multi_step',   'Calculate: 5 + 3 * 2',                                                        '11'),
    ('percentage',   'What is 25% of 80?',                                                          '20'),
    ('ratio',        'Two quantities are in the ratio 3:2. If the total is 50, what is the first quantity?', '30'),
    ('algebra',      'Solve for x: 2x + 4 = 10',                                                   '3'),
    ('word_problem', 'Alice has 12 apples. She buys 7 more. How many apples does she have now?',    '19'),
]

LABELS  = ['init', 'early', 'mid', 'final', 'best']
avail   = [lb for lb in LABELS if os.path.exists(os.path.join(CK_DIR, f'{lb}.pt'))]
print('Progression checkpoints found:', avail)

if len(avail) < 2:
    print('Only best.pt available — progression comparison requires init/early/mid/final.')
    print('These are saved during training. Run training first to generate them.')
else:
    progression = {}
    for lb in avail:
        m2, t2, ck2 = load_model_and_tokenizer(
            os.path.join(CK_DIR, f'{lb}.pt'), device=DEVICE)
        step2 = ck2.get('global_step', '?')
        progression[lb] = {'_step': step2}
        for fam, prob, exp in PROBE:
            res = generate_answer(prob, m2, t2, DEVICE, True, 64)
            try: en = float(exp)
            except ValueError: en = 0.0
            vr = verify(res['full_text'], exp, en)
            progression[lb][prob] = {'correct': vr['correct'],
                                      'extracted': res['extracted_answer']}
        del m2
        if DEVICE == 'cuda': torch.cuda.empty_cache()
        nc = sum(1 for _, p, _ in PROBE if progression[lb].get(p, {}).get('correct'))
        print(f'  [{lb}] step={step2}  correct={nc}/{len(PROBE)}')

    print()
    hdr = f'{"Family":<14}{"Exp":<6}'
    for lb in avail: hdr += f'  {lb[:5]}(s{str(progression[lb]["_step"])[:4]})'.ljust(12)
    print(hdr)
    print('-' * len(hdr))
    for fam, prob, exp in PROBE:
        row = f'{fam:<14}{exp:<6}'
        for lb in avail:
            d = progression[lb].get(prob, {})
            mk = 'OK' if d.get('correct') else '--'
            row += f'  {mk}({str(d.get("extracted","?"))[:5]})'.ljust(12)
        print(row)

## Step 8 — Full outputs from best.pt

In [ ]:
print('=== Full model outputs (best.pt) — correct and incorrect ===')
print()
for fam, prob, exp in PROBE:
    res = generate_answer(prob, model, tokenizer, DEVICE, True, 64)
    try: en = float(exp)
    except ValueError: en = 0.0
    vr = verify(res['full_text'], exp, en)
    status = 'CORRECT' if vr['correct'] else 'WRONG  '
    print(f'[{status}] [{fam}]  expected={exp}, got={vr["predicted"]}')
    print(res['full_text'])
    print()

## Step 9 — Summary

In [ ]:
print('=' * 60)
print('EVALUATION SUMMARY')
print('=' * 60)
print(f'Checkpoint:  {BEST_CK}')
print(f'Parameters:  {n_params:,}')
print(f'Step:        {step}')
print()
print('Test benchmark (500 problems):')
print(f'  Overall: {bench.correct}/{bench.total} = {bench.accuracy:.1%}')
for fam, acc in sorted(bench.family_accuracy.items(), key=lambda x: -x[1]):
    bar = '#' * int(acc * 25)
    print(f'  {fam:<25} {acc:>6.1%}  {bar}')
print()
print('Generalization (200/level):')
for lv in sorted(gen_results):
    r = gen_results[lv]
    print(f'  Level {lv}: {r["correct"]:>3}/{r["total"]} = {r["accuracy"]:.1%}  {LEVEL_DESC[lv]}')
print(f'  Overall: {tc}/{tn} = {tc/tn:.1%}')
print()
zero = [f for f, a in bench.family_accuracy.items() if a == 0.0]
if zero:
    print(f'Zero-accuracy families: {zero}')
print('=' * 60)

## Step 10 — Interactive inference

Type math problems and press Enter. Type `exit` to stop.

The model currently achieves ~17.8% overall accuracy.  
It works best on number sequences and simple comparisons.  
Multi-step arithmetic and percentages are mostly wrong.

In [21]:
from inference.generate import interactive_inference

interactive_inference(
    checkpoint_path=BEST_CK,
    device=DEVICE,
    max_new_tokens=128,
    temperature=0.0,
    verify_answers=True,
)